## PARTE 2 — Regresión Lineal Simple + Predicción
**Proyecto:** Monitor Inteligente de Asistencia + EmoScan | Semana 12 | SENATI

Predice el **puntaje de asistencia** (0–100) usando datos del Monitor + EmoScan.

### Celda 1 — Importaciones

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib joblib  # ← descomentar en Colab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
print('✅ Librerías importadas')

### Celda 2 — Carga del dataset

In [ ]:
df = pd.read_csv('dataset_asistencia.csv')
print(f'📊 {df.shape[0]} filas × {df.shape[1]} columnas')
print(df.head())
print(df.describe().round(3))

### Celda 3 — Exploración visual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].hist(df['puntaje_asistencia'], bins=20, color='#2196F3', edgecolor='white')
axes[0].set_title('Distribución del Puntaje')
axes[1].scatter(df['confianza'], df['puntaje_asistencia'], alpha=0.5, color='#4CAF50')
axes[1].set_title('Confianza vs Puntaje')
axes[2].scatter(df['hora_llegada'], df['puntaje_asistencia'], alpha=0.5, color='#FF5722')
axes[2].set_title('Hora Llegada vs Puntaje')
plt.tight_layout()
plt.savefig('graficos_exploracion.png', dpi=150, bbox_inches='tight')
plt.show()

### Celda 4 — Regresión Lineal Simple (1 variable: confianza → puntaje)

In [ ]:
X_simple = df[['confianza']]
y = df['puntaje_asistencia']
X_tr, X_te, y_tr, y_te = train_test_split(X_simple, y, test_size=0.2, random_state=42)
modelo_simple = LinearRegression().fit(X_tr, y_tr)
y_pred_s = modelo_simple.predict(X_te)
r2_s  = r2_score(y_te, y_pred_s)
mae_s = mean_absolute_error(y_te, y_pred_s)
mse_s = mean_squared_error(y_te, y_pred_s)
print(f'Ecuación: puntaje = {modelo_simple.coef_[0]:.2f} × confianza + {modelo_simple.intercept_:.2f}')
print(f'R²={r2_s:.4f} | MAE={mae_s:.4f} | MSE={mse_s:.4f}')

### Celda 5 — Gráfico regresión simple

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(X_te, y_te, alpha=0.6, label='Datos reales', color='#2196F3')
x_l = np.linspace(X_simple.min().values[0], X_simple.max().values[0], 100).reshape(-1,1)
ax.plot(x_l, modelo_simple.predict(x_l), color='#F44336', lw=2.5, label=f'Regresión (R²={r2_s:.3f})')
ax.set_xlabel('Confianza facial'); ax.set_ylabel('Puntaje Asistencia')
ax.set_title('Regresión Lineal Simple: Confianza → Puntaje Asistencia')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('regresion_simple.png', dpi=150, bbox_inches='tight')
plt.show()

### Celda 6 — Regresión Lineal Múltiple (todas las features)

In [ ]:
features = ['dia_semana','hora_llegada','confianza','liveness_score','happy','neutral','sad','angry']
X_m = df[features]
X_trm, X_tem, y_trm, y_tem = train_test_split(X_m, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_trs = scaler.fit_transform(X_trm)
X_tes = scaler.transform(X_tem)
modelo_multi = LinearRegression().fit(X_trs, y_trm)
y_pred_m = modelo_multi.predict(X_tes)
r2_m  = r2_score(y_tem, y_pred_m)
mae_m = mean_absolute_error(y_tem, y_pred_m)
mse_m = mean_squared_error(y_tem, y_pred_m)
print(f'R²={r2_m:.4f} | MAE={mae_m:.4f} | MSE={mse_m:.4f}')
coef_df = pd.DataFrame({'Feature':features,'Coef':modelo_multi.coef_.round(4)}).sort_values('Coef',ascending=False)
print(coef_df.to_string(index=False))

### Celda 7 — Gráfico predicción vs real

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(y_tem, y_pred_m, alpha=0.6, color='#9C27B0')
ax.plot([y.min(),y.max()],[y.min(),y.max()],'r--',lw=2,label='Predicción perfecta')
ax.set_xlabel('Puntaje Real'); ax.set_ylabel('Puntaje Predicho')
ax.set_title(f'Predicción vs Real — Regresión Múltiple (R²={r2_m:.3f})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('prediccion_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

### Celda 8 — Exportar modelo y coeficientes

In [ ]:
joblib.dump(modelo_multi, 'modelo_regresion.pkl')
joblib.dump(scaler,       'scaler_regresion.pkl')
with open('coef.json','w',encoding='utf-8') as f:
    json.dump({
        'modelo':'Regresión Lineal Múltiple','features':features,
        'coeficientes':modelo_multi.coef_.tolist(),'intercepto':float(modelo_multi.intercept_),
        'metricas':{'r2_simple':round(r2_s,4),'mae_simple':round(mae_s,4),
                    'r2_multiple':round(r2_m,4),'mae_multiple':round(mae_m,4)}
    }, f, indent=2, ensure_ascii=False)
print('✅ Exportados: modelo_regresion.pkl | scaler_regresion.pkl | coef.json')

### Celda 9 — Predicción de ejemplo

In [ ]:
ejemplo = pd.DataFrame([{'dia_semana':0,'hora_llegada':482,'confianza':0.92,
    'liveness_score':0.88,'happy':0.55,'neutral':0.30,'sad':0.10,'angry':0.05}])
pred = float(np.clip(modelo_multi.predict(scaler.transform(ejemplo))[0], 0, 100))
estado = 'PRESENTE ✅' if pred>=75 else ('TARDANZA ⚠️' if pred>=45 else 'AUSENTE ❌')
print(f'Puntaje predicho: {pred:.2f} / 100')
print(f'Estado inferido : {estado}')